#Monte-Carlo without using Gym

In [1]:
import numpy as np

In [2]:
class Gridworld:
    def __init__(self):
        self.Q = None
        self.actionSpace = ('U', 'D', 'L', 'R')

        self.actions = {
            (i, j): [
                a
                for a in (
                    'U' if i > 0 else '',
                    'D' if i < 3 else '',
                    'L' if j > 0 else '',
                    'R' if j < 3 else '',
                )
                if a != ''
            ]
            for i in range(4)
            for j in range(4)
        }
        self.rewards = {(3,3): 0.03, (1,3): -0.01, (2,1): -0.01, (3,1): -0.01}
        self.terminal_states = set(self.rewards.keys())

        self.explored = 0
        self.exploited = 0
        self.initialQ()

    def initialQ(self):
        self.Q = {}
        for state in self.actions:
            self.Q[state] = {}
            for move in self.actions[state]:
                self.Q[state][move] = 0

    def updateQ(self, new_val_q, update_val = 0.05):
        for state in self.Q:
            for action in self.Q[state]:
                self.Q[state][action] += update_val * (new_val_q[state][action] - self.Q[state][action])
    
    def random_policy(self):
        policy = {}
        for state in self.actions:
            policy[state] = np.random.choice(self.actions[state])
        return policy

    def reset(self):
        return (0,0)

    def terminal(self, s):
        return s in self.terminal_states

    def new_state(self, state, action):
        row, column = state

        if action == 'U':
            row -= 1
        elif action == 'D':
            row += 1
        elif action == 'L':
            column -= 1
        elif action == 'R':
            column += 1

        return row, column

    def choose_action(self, state, policy, explore_rate):
        if explore_rate > np.random.rand():
            self.explored += 1
            return np.random.choice(self.actions[state])
        self.exploited += 1
        return policy[state]

    def move(self, state, policy, explore_rate):
        action = self.choose_action(state, policy, explore_rate)
        row, column = self.new_state(state, action)
        if(row, column) in self.rewards:
            return action, (row, column), self.rewards[(row, column)]
        return action, (row, column), 0

    def print_policy(self, policy):
        print("--------------------------------")
        line = ""
        counter = 0
        for item in policy:
            if item in self.terminal_states:
                line += " | T |"
            else:
                line += f" | {policy[item]} |"
            counter += 1
            if counter > 3:
                print(line)
                print("--------------------------------")
                counter = 0
                line = ""

In [3]:
env = Gridworld()
policy = env.random_policy()
print("Initial Random Policy:")
env.print_policy(policy)

Initial Random Policy:
--------------------------------
 | R | | D | | L | | D |
--------------------------------
 | D | | D | | D | | T |
--------------------------------
 | U | | T | | R | | L |
--------------------------------
 | U | | T | | U | | T |
--------------------------------


In [4]:
for i in range(1001):
    estimated_q = {
        state: {action: 0.0 for action in env.Q[state]} for state in env.Q
    }

    visit_counts = {
        state: {action: 0 for action in env.Q[state]} for state in env.Q
    }

    for j in range(1000):
        trajectory = []
        state = env.reset()
        step_count = 0

        while(not env.terminal(state)) and (step_count<30):
            action, new_state, reward = env.move(state, policy, explore_rate=0.05)
            trajectory.append(((state, action), reward))
            state = new_state
            step_count += 1

        G = 0.0

        for (state, action), reward in reversed(trajectory):
            G = reward + 0.9 * G
            visit_counts[state][action] += 1
            n = visit_counts[state][action]
            estimated_q[state][action] += (1 / n) * (
                G - estimated_q[state][action]
            )

    env.updateQ(estimated_q)

    for state in policy:
        policy[state] = max(env.Q[state], key = env.Q[state].get)

    if (i%100) == 0:
        print(f"\nStep: {i}")
        env.print_policy(policy)
        print(f"Exploited: {env.exploited}, Explored: {env.explored}")


Step: 0
--------------------------------
 | D | | R | | D | | D |
--------------------------------
 | D | | L | | U | | T |
--------------------------------
 | D | | T | | D | | D |
--------------------------------
 | R | | T | | U | | T |
--------------------------------
Exploited: 3860, Explored: 196

Step: 100
--------------------------------
 | R | | D | | D | | L |
--------------------------------
 | R | | R | | D | | T |
--------------------------------
 | U | | T | | R | | D |
--------------------------------
 | U | | T | | U | | T |
--------------------------------
Exploited: 622000, Explored: 32903

Step: 200
--------------------------------
 | R | | R | | D | | L |
--------------------------------
 | R | | R | | D | | T |
--------------------------------
 | U | | T | | R | | D |
--------------------------------
 | U | | T | | U | | T |
--------------------------------
Exploited: 1208153, Explored: 63849

Step: 300
--------------------------------
 | R | | D | | D | | L |
---